In [ ]:
from pathlib import Path
from datetime import datetime
import json
import time

import requests
import pandas as pd


BASE_URL = "https://dadosabertos.camara.leg.br/api/v2"

RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


def salvar_json(nome_arquivo: str, conteudo: dict | list) -> None:
    """
    Salva o retorno bruto da API em JSON.
    """
    caminho = RAW_DIR / nome_arquivo

    with open(caminho, "w", encoding="utf-8") as arquivo:
        json.dump(conteudo, arquivo, ensure_ascii=False, indent=2)


def get_api(endpoint: str, params: dict | None = None) -> dict:
    """
    Faz uma requisição GET simples na API da Câmara.
    """
    url = f"{BASE_URL}/{endpoint}"

    headers = {
        "Accept": "application/json"
    }

    response = requests.get(
        url,
        params=params,
        headers=headers,
        timeout=30
    )

    response.raise_for_status()

    return response.json()


def get_paginated(endpoint: str, params: dict | None = None, itens: int = 100) -> list[dict]:
    """
    Consome endpoints paginados da API da Câmara.
    A API retorna os dados dentro da chave 'dados'.
    """
    todos_registros = []
    pagina = 1

    params_base = params.copy() if params else {}
    params_base["itens"] = itens

    while True:
        params_base["pagina"] = pagina

        print(f"Extraindo {endpoint} | página {pagina}")

        resposta = get_api(endpoint, params=params_base)
        dados = resposta.get("dados", [])

        if not dados:
            break

        todos_registros.extend(dados)

        salvar_json(
            nome_arquivo=f"{endpoint.replace('/', '_')}_pagina_{pagina}.json",
            conteudo=resposta
        )

        if len(dados) < itens:
            break

        pagina += 1
        time.sleep(0.3)

    return todos_registros


def criar_dataframe(registros: list[dict]) -> pd.DataFrame:
    """
    Cria DataFrame padronizado a partir de uma lista de dicionários.
    """
    df = pd.DataFrame(registros)

    if df.empty:
        return df

    df.columns = (
        df.columns
        .str.strip()
        .str.replace(" ", "_")
        .str.replace("-", "_")
        .str.lower()
    )

    return df


def extrair_deputados() -> pd.DataFrame:
    registros = get_paginated("deputados")

    df = criar_dataframe(registros)

    colunas = [
        "id",
        "nome",
        "siglapartido",
        "siglauf",
        "idlegislatura",
        "urlfoto",
        "email"
    ]

    colunas_existentes = [col for col in colunas if col in df.columns]

    return df[colunas_existentes]


def extrair_partidos() -> pd.DataFrame:
    registros = get_paginated("partidos")

    df = criar_dataframe(registros)

    colunas = [
        "id",
        "sigla",
        "nome",
        "uri"
    ]

    colunas_existentes = [col for col in colunas if col in df.columns]

    return df[colunas_existentes]


def extrair_proposicoes(data_inicio: str, data_fim: str) -> pd.DataFrame:
    params = {
        "dataInicio": data_inicio,
        "dataFim": data_fim,
        "ordem": "ASC",
        "ordenarPor": "id"
    }

    registros = get_paginated("proposicoes", params=params)

    df = criar_dataframe(registros)

    colunas = [
        "id",
        "siglatipo",
        "codtipo",
        "numero",
        "ano",
        "ementa",
        "dataapresentacao",
        "uri"
    ]

    colunas_existentes = [col for col in colunas if col in df.columns]

    return df[colunas_existentes]


def extrair_votacoes(data_inicio: str | None = None, data_fim: str | None = None) -> pd.DataFrame:
    params = {
        "ordem": "ASC",
        "ordenarPor": "dataHoraRegistro"
    }

    if data_inicio:
        params["dataInicio"] = data_inicio

    if data_fim:
        params["dataFim"] = data_fim

    registros = get_paginated("votacoes", params=params)

    df = criar_dataframe(registros)

    colunas = [
        "id",
        "uri",
        "data",
        "datahoraregistro",
        "siglaorgao",
        "uriOrgao",
        "descricao",
        "aprovacao"
    ]

    colunas_existentes = [col for col in colunas if col.lower() in df.columns]

    return df


def extrair_temas_proposicoes() -> pd.DataFrame:
    resposta = get_api("referencias/proposicoes/codTema")

    salvar_json(
        nome_arquivo="referencias_proposicoes_codTema.json",
        conteudo=resposta
    )

    return criar_dataframe(resposta.get("dados", []))


def extrair_situacoes_proposicao() -> pd.DataFrame:
    resposta = get_api("referencias/situacoesProposicao")

    salvar_json(
        nome_arquivo="referencias_situacoesProposicao.json",
        conteudo=resposta
    )

    return criar_dataframe(resposta.get("dados", []))


def salvar_dataframe(df: pd.DataFrame, nome_tabela: str) -> None:
    """
    Salva DataFrame em CSV e Parquet.
    """
    caminho_csv = PROCESSED_DIR / f"{nome_tabela}.csv"
    caminho_parquet = PROCESSED_DIR / f"{nome_tabela}.parquet"

    df.to_csv(caminho_csv, index=False, encoding="utf-8")
    df.to_parquet(caminho_parquet, index=False)

    print(f"Salvo: {caminho_csv}")
    print(f"Salvo: {caminho_parquet}")


def main():
    data_inicio = "2026-04-01"
    data_fim = "2026-04-30"

    print("Extraindo deputados...")
    df_deputados = extrair_deputados()

    print("Extraindo partidos...")
    df_partidos = extrair_partidos()

    print("Extraindo proposições...")
    df_proposicoes = extrair_proposicoes(data_inicio, data_fim)

    print("Extraindo votações...")
    df_votacoes = extrair_votacoes(data_inicio, data_fim)

    print("Extraindo temas...")
    df_temas = extrair_temas_proposicoes()

    print("Extraindo situações das proposições...")
    df_situacoes = extrair_situacoes_proposicao()

    salvar_dataframe(df_deputados, "dim_deputado")
    salvar_dataframe(df_partidos, "dim_partido")
    salvar_dataframe(df_proposicoes, "fato_proposicao_base")
    salvar_dataframe(df_votacoes, "fato_votacao")
    salvar_dataframe(df_temas, "dim_tema")
    salvar_dataframe(df_situacoes, "dim_situacao")

    print("\nResumo dos DataFrames:")
    print(f"Deputados: {df_deputados.shape}")
    print(f"Partidos: {df_partidos.shape}")
    print(f"Proposições: {df_proposicoes.shape}")
    print(f"Votações: {df_votacoes.shape}")
    print(f"Temas: {df_temas.shape}")
    print(f"Situações: {df_situacoes.shape}")

    print("\nPrévia de proposições:")
    print("\nResumo dos DataFrames:")
    resumo = pd.DataFrame({
        "tabela": [
            "dim_deputado",
            "dim_partido",
            "fato_proposicao_base",
            "fato_votacao",
            "dim_tema",
            "dim_situacao"
        ],
        "linhas": [
            len(df_deputados),
            len(df_partidos),
            len(df_proposicoes),
            len(df_votacoes),
            len(df_temas),
            len(df_situacoes)
        ],
        "colunas": [
            len(df_deputados.columns),
            len(df_partidos.columns),
            len(df_proposicoes.columns),
            len(df_votacoes.columns),
            len(df_temas.columns),
            len(df_situacoes.columns)
        ]
    })

    print(resumo.to_string(index=False))

    print("\nColunas por DataFrame:")
    print("dim_deputado:", list(df_deputados.columns))
    print("dim_partido:", list(df_partidos.columns))
    print("fato_proposicao_base:", list(df_proposicoes.columns))
    print("fato_votacao:", list(df_votacoes.columns))
    print("dim_tema:", list(df_temas.columns))
    print("dim_situacao:", list(df_situacoes.columns))


if __name__ == "__main__":
    main()

In [1]:
from pathlib import Path

import pandas as pd


PROCESSED_DIR = Path("data/processed")
MODEL_DIR = Path("data/model")

MODEL_DIR.mkdir(parents=True, exist_ok=True)


def carregar_csv(nome_arquivo: str) -> pd.DataFrame:
    caminho = PROCESSED_DIR / nome_arquivo
    return pd.read_csv(caminho)


def salvar_tabela(df: pd.DataFrame, nome_tabela: str) -> None:
    caminho_csv = MODEL_DIR / f"{nome_tabela}.csv"
    caminho_parquet = MODEL_DIR / f"{nome_tabela}.parquet"

    df.to_csv(caminho_csv, index=False, encoding="utf-8")
    df.to_parquet(caminho_parquet, index=False)

    print(f"Salvo: {caminho_csv}")
    print(f"Salvo: {caminho_parquet}")


def tratar_texto(df: pd.DataFrame, colunas: list[str]) -> pd.DataFrame:
    for coluna in colunas:
        if coluna in df.columns:
            df[coluna] = df[coluna].fillna("").astype(str).str.strip()
    return df


def modelar_dim_deputado() -> pd.DataFrame:
    df = carregar_csv("dim_deputado.csv")

    df = df.rename(columns={
        "id": "id_deputado",
        "siglapartido": "sigla_partido",
        "siglauf": "sigla_uf",
        "idlegislatura": "id_legislatura",
        "urlfoto": "url_foto"
    })

    df = tratar_texto(df, [
        "nome",
        "sigla_partido",
        "sigla_uf",
        "url_foto",
        "email"
    ])

    df["id_deputado"] = pd.to_numeric(df["id_deputado"], errors="coerce").astype("Int64")
    df["id_legislatura"] = pd.to_numeric(df["id_legislatura"], errors="coerce").astype("Int64")

    return df.drop_duplicates(subset=["id_deputado"])


def modelar_dim_partido() -> pd.DataFrame:
    df = carregar_csv("dim_partido.csv")

    df = df.rename(columns={
        "id": "id_partido"
    })

    df = tratar_texto(df, [
        "sigla",
        "nome",
        "uri"
    ])

    df["id_partido"] = pd.to_numeric(df["id_partido"], errors="coerce").astype("Int64")

    return df.drop_duplicates(subset=["id_partido"])


def modelar_dim_tema() -> pd.DataFrame:
    df = carregar_csv("dim_tema.csv")

    df = df.rename(columns={
        "cod": "cod_tema"
    })

    df = tratar_texto(df, [
        "sigla",
        "nome",
        "descricao"
    ])

    df["cod_tema"] = pd.to_numeric(df["cod_tema"], errors="coerce").astype("Int64")

    return df.drop_duplicates(subset=["cod_tema"])


def modelar_dim_situacao() -> pd.DataFrame:
    df = carregar_csv("dim_situacao.csv")

    df = df.rename(columns={
        "cod": "cod_situacao"
    })

    df = tratar_texto(df, [
        "sigla",
        "nome",
        "descricao"
    ])

    df["cod_situacao"] = pd.to_numeric(df["cod_situacao"], errors="coerce").astype("Int64")

    return df.drop_duplicates(subset=["cod_situacao"])


def modelar_fato_proposicao() -> pd.DataFrame:
    df = carregar_csv("fato_proposicao_base.csv")

    df = df.rename(columns={
        "id": "id_proposicao",
        "siglatipo": "sigla_tipo",
        "codtipo": "cod_tipo",
        "dataapresentacao": "data_apresentacao"
    })

    df = tratar_texto(df, [
        "sigla_tipo",
        "ementa",
        "uri"
    ])

    df["id_proposicao"] = pd.to_numeric(df["id_proposicao"], errors="coerce").astype("Int64")
    df["cod_tipo"] = pd.to_numeric(df["cod_tipo"], errors="coerce").astype("Int64")
    df["numero"] = pd.to_numeric(df["numero"], errors="coerce").astype("Int64")
    df["ano"] = pd.to_numeric(df["ano"], errors="coerce").astype("Int64")
    df["data_apresentacao"] = pd.to_datetime(df["data_apresentacao"], errors="coerce").dt.date


    return df.drop_duplicates(subset=["id_proposicao"])


def modelar_fato_votacao() -> pd.DataFrame:
    df = carregar_csv("fato_votacao.csv")

    df = df.rename(columns={
        "id": "id_votacao",
        "datahoraregistro": "data_hora_registro",
        "siglaorgao": "sigla_orgao",
        "uriorgao": "uri_orgao",
        "urievento": "uri_evento",
        "proposicaoobjeto": "proposicao_objeto",
        "uriproposicaoobjeto": "uri_proposicao_objeto"
    })

    df = tratar_texto(df, [
        "id_votacao",
        "uri",
        "sigla_orgao",
        "uri_orgao",
        "uri_evento",
        "proposicao_objeto",
        "uri_proposicao_objeto",
        "descricao",
        "aprovacao"
    ])

    if "data" in df.columns:
        df["data"] = pd.to_datetime(df["data"], errors="coerce").dt.date

    if "data_hora_registro" in df.columns:
        df["data_hora_registro"] = pd.to_datetime(df["data_hora_registro"], errors="coerce")

    return df.drop_duplicates(subset=["id_votacao"])


def main():
    dim_deputado = modelar_dim_deputado()
    dim_partido = modelar_dim_partido()
    dim_tema = modelar_dim_tema()
    dim_situacao = modelar_dim_situacao()
    fato_proposicao = modelar_fato_proposicao()
    fato_votacao = modelar_fato_votacao()

    salvar_tabela(dim_deputado, "dim_deputado")
    salvar_tabela(dim_partido, "dim_partido")
    salvar_tabela(dim_tema, "dim_tema")
    salvar_tabela(dim_situacao, "dim_situacao")
    salvar_tabela(fato_proposicao, "fato_proposicao")
    salvar_tabela(fato_votacao, "fato_votacao")

    resumo = pd.DataFrame({
        "tabela": [
            "dim_deputado",
            "dim_partido",
            "dim_tema",
            "dim_situacao",
            "fato_proposicao",
            "fato_votacao"
        ],
        "linhas": [
            len(dim_deputado),
            len(dim_partido),
            len(dim_tema),
            len(dim_situacao),
            len(fato_proposicao),
            len(fato_votacao)
        ],
        "colunas": [
            len(dim_deputado.columns),
            len(dim_partido.columns),
            len(dim_tema.columns),
            len(dim_situacao.columns),
            len(fato_proposicao.columns),
            len(fato_votacao.columns)
        ]
    })

    print("\nResumo das tabelas modeladas:")
    print(resumo.to_string(index=False))

    print("\nColunas finais:")
    print("dim_deputado:", list(dim_deputado.columns))
    print("dim_partido:", list(dim_partido.columns))
    print("dim_tema:", list(dim_tema.columns))
    print("dim_situacao:", list(dim_situacao.columns))
    print("fato_proposicao:", list(fato_proposicao.columns))
    print("fato_votacao:", list(fato_votacao.columns))


if __name__ == "__main__":
    main()

Salvo: data\model\dim_deputado.csv
Salvo: data\model\dim_deputado.parquet
Salvo: data\model\dim_partido.csv
Salvo: data\model\dim_partido.parquet
Salvo: data\model\dim_tema.csv
Salvo: data\model\dim_tema.parquet
Salvo: data\model\dim_situacao.csv
Salvo: data\model\dim_situacao.parquet
Salvo: data\model\fato_proposicao.csv
Salvo: data\model\fato_proposicao.parquet
Salvo: data\model\fato_votacao.csv
Salvo: data\model\fato_votacao.parquet

Resumo das tabelas modeladas:
         tabela  linhas  colunas
   dim_deputado     513        7
    dim_partido      21        4
       dim_tema      32        4
   dim_situacao      99        4
fato_proposicao   15951        8
   fato_votacao    1393       11

Colunas finais:
dim_deputado: ['id_deputado', 'nome', 'sigla_partido', 'sigla_uf', 'id_legislatura', 'url_foto', 'email']
dim_partido: ['id_partido', 'sigla', 'nome', 'uri']
dim_tema: ['cod_tema', 'sigla', 'nome', 'descricao']
dim_situacao: ['cod_situacao', 'sigla', 'nome', 'descricao']
fato_prop

In [3]:
import os
from dotenv import load_dotenv
from supabase import create_client


load_dotenv()

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_SERVICE_ROLE_KEY = os.getenv("SUPABASE_SERVICE_ROLE_KEY")

if not SUPABASE_URL:
    raise ValueError("SUPABASE_URL não encontrada no .env")

if not SUPABASE_SERVICE_ROLE_KEY:
    raise ValueError("SUPABASE_SERVICE_ROLE_KEY não encontrada no .env")

supabase = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)

res = (
    supabase
    .schema("radar")
    .table("dim_partido")
    .select("*")
    .limit(1)
    .execute()
)

print("Conexão OK com Supabase.")
print(res.data)

Conexão OK com Supabase.
[]


In [4]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from supabase import create_client


load_dotenv(override=True)

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_SERVICE_ROLE_KEY = os.getenv("SUPABASE_SERVICE_ROLE_KEY")

MODEL_DIR = Path("data/model")
SCHEMA = "radar"

if not SUPABASE_URL:
    raise ValueError("SUPABASE_URL não encontrada no .env")

if not SUPABASE_SERVICE_ROLE_KEY:
    raise ValueError("SUPABASE_SERVICE_ROLE_KEY não encontrada no .env")


supabase = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)


def limpar_valores_para_json(df: pd.DataFrame) -> pd.DataFrame:
    """
    Converte valores incompatíveis com JSON/PostgREST:
    - NaN/NaT/pd.NA -> None
    - datas/timestamps -> string ISO
    """
    df = df.copy()

    for coluna in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[coluna]):
            df[coluna] = df[coluna].dt.strftime("%Y-%m-%d %H:%M:%S")

    df = df.astype(object)
    df = df.where(pd.notnull(df), None)

    return df


def carregar_csv(nome_arquivo: str) -> pd.DataFrame:
    caminho = MODEL_DIR / nome_arquivo
    df = pd.read_csv(caminho)

    return limpar_valores_para_json(df)


def inserir_em_lotes(nome_tabela: str, df: pd.DataFrame, lote: int = 500) -> None:
    total = len(df)

    print(f"\nCarregando {nome_tabela}: {total} linhas")

    for inicio in range(0, total, lote):
        fim = min(inicio + lote, total)
        registros = df.iloc[inicio:fim].to_dict(orient="records")

        (
            supabase
            .schema(SCHEMA)
            .table(nome_tabela)
            .upsert(registros)
            .execute()
        )

        print(f"{nome_tabela}: linhas {inicio + 1} até {fim} carregadas")


def main():
    ordem_carga = [
        ("dim_deputado", "dim_deputado.csv"),
        ("dim_partido", "dim_partido.csv"),
        ("dim_tema", "dim_tema.csv"),
        ("dim_situacao", "dim_situacao.csv"),
        ("fato_proposicao", "fato_proposicao.csv"),
        ("fato_votacao", "fato_votacao.csv"),
    ]

    for nome_tabela, nome_arquivo in ordem_carga:
        df = carregar_csv(nome_arquivo)
        inserir_em_lotes(nome_tabela, df)

    print("\nCarga finalizada com sucesso.")


if __name__ == "__main__":
    main()


Carregando dim_deputado: 513 linhas
dim_deputado: linhas 1 até 500 carregadas
dim_deputado: linhas 501 até 513 carregadas

Carregando dim_partido: 21 linhas
dim_partido: linhas 1 até 21 carregadas

Carregando dim_tema: 32 linhas
dim_tema: linhas 1 até 32 carregadas

Carregando dim_situacao: 99 linhas
dim_situacao: linhas 1 até 99 carregadas

Carregando fato_proposicao: 15951 linhas
fato_proposicao: linhas 1 até 500 carregadas
fato_proposicao: linhas 501 até 1000 carregadas
fato_proposicao: linhas 1001 até 1500 carregadas
fato_proposicao: linhas 1501 até 2000 carregadas
fato_proposicao: linhas 2001 até 2500 carregadas
fato_proposicao: linhas 2501 até 3000 carregadas
fato_proposicao: linhas 3001 até 3500 carregadas
fato_proposicao: linhas 3501 até 4000 carregadas
fato_proposicao: linhas 4001 até 4500 carregadas
fato_proposicao: linhas 4501 até 5000 carregadas
fato_proposicao: linhas 5001 até 5500 carregadas
fato_proposicao: linhas 5501 até 6000 carregadas
fato_proposicao: linhas 6001 at